# Step 14 -- CARE-Sim Selected-Set World Model

Trains the new CARE-Sim track on the Step 10f selected replay dataset.

Selected-set design:
- 6 dynamic states: `Hb, BUN, Creatinine, Phosphate, HR, Chloride`
- 3 static confounders: `age, charlson_score, prior_ed_visits_6m`
- 5 actions: `vasopressor, ivfluid, antibiotic, diuretic, mechvent`
- no reward head; transition + terminal only
- optional FCI-based causal action mask controlled by a notebook flag

## Before running

**Step 1:** From the repo root, rebuild the upload zip locally:
```
python scripts/prepare_colab_upload.py
```

**Step 2:** Upload to Google Drive:
- `caresim_colab.zip` -> `MyDrive/CareAI/`
- `data/processed/icu_readmit/rl_dataset_selected.parquet` -> `MyDrive/CareAI/data/`

**Step 3:** Runtime -> Change runtime type -> **T4 GPU**.


In [ ]:
# Cell 1: Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be much slower on CPU.')


In [ ]:
# Cell 2: Unzip source files + verify data
import os, sys, zipfile

USE_CAUSAL = True   # set False for unconstrained selected-set training

DRIVE_ROOT = '/content/drive/MyDrive/CareAI'
ZIP_PATH   = os.path.join(DRIVE_ROOT, 'caresim_colab.zip')
UNZIP_DIR  = '/content/caresim_src'
SRC_PATH   = os.path.join(UNZIP_DIR, 'src')
DATA_PATH  = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_selected.parquet')
MODEL_SUBDIR = 'caresim_selected_causal' if USE_CAUSAL else 'caresim_selected'
MODEL_DIR  = os.path.join(DRIVE_ROOT, 'models', 'icu_readmit', MODEL_SUBDIR)

SMOKE_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_14_caresim_smoke_test.py')
TRAIN_SCRIPT = os.path.join(UNZIP_DIR, 'scripts', 'icu_readmit', 'step_14_caresim_train_selected.py')

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'caresim_colab.zip not found at {ZIP_PATH}. Run prepare_colab_upload.py locally first.')

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(UNZIP_DIR)
print(f'Unzipped to {UNZIP_DIR}')

for pkg_dir in [
    os.path.join(SRC_PATH, 'careai'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit'),
    os.path.join(SRC_PATH, 'careai', 'icu_readmit', 'caresim'),
]:
    init = os.path.join(pkg_dir, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Data file not found:\n  {DATA_PATH}\n'
        'Upload rl_dataset_selected.parquet to MyDrive/CareAI/data/'
    )

size_gb = os.path.getsize(DATA_PATH) / 1e9
print(f'Data file OK ({size_gb:.2f} GB)')
print(f'Causal constraints: {USE_CAUSAL}')
print(f'Model output: {MODEL_DIR}')
print('Ready.')


In [ ]:
# Cell 3: Smoke test (synthetic data)
import subprocess

def run_streaming(cmd, env):
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

rc = run_streaming(
    [sys.executable, '-u', SMOKE_SCRIPT],
    env={**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'},
)
if rc != 0:
    raise RuntimeError('Smoke test FAILED -- do not proceed to full training.')
print('Smoke test PASSED.')


In [ ]:
# Cell 4: Full training
BASE_ARGS = [
    sys.executable, '-u', TRAIN_SCRIPT,
    '--data',        DATA_PATH,
    '--save-dir',    MODEL_DIR,
    '--device',      device,
    '--n-models',    '5',
    '--d-model',     '256',
    '--n-heads',     '8',
    '--n-layers',    '4',
    '--n-epochs',    '30',
    '--batch-size',  '64',
    '--lr',          '1e-3',
]

if USE_CAUSAL:
    BASE_ARGS.append('--causal-constraints')

ENV = {**os.environ, 'PYTHONPATH': SRC_PATH, 'PYTHONUNBUFFERED': '1'}
print('=' * 60)
label = 'selected-set causal CARE-Sim' if USE_CAUSAL else 'selected-set CARE-Sim'
print(f'FULL TRAINING -- {label} (9-state / 5-action, no reward head)')
print('=' * 60)
rc = run_streaming(BASE_ARGS, env=ENV)
if rc != 0:
    raise RuntimeError('Selected-set training FAILED.')
print('\nSelected-set training complete.')


In [ ]:
# Cell 5: Verify outputs
print('Files saved to Drive:')
for root, dirs, files in os.walk(MODEL_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, MODEL_DIR)
        print(f'  {rel:55s}  {size_kb:8.1f} KB')

print(f'\nDownload models/icu_readmit/{MODEL_SUBDIR}/ from Drive to:')
print(f'  CareAI/models/icu_readmit/{MODEL_SUBDIR}/')
